# Milestone 1 lab, notebook 2 of 7: Read every order

## Where we are

| Notebook | What you do |
|---|---|
| 1. The question and the source | Read the question. Meet the order system. Look at one order |
| **2. Read every order** (you are here) | Follow the pages of the API to the end |
| 3. The answer by hand | Work out the answer with plain Python. Meet the time zone trap |
| 4. Transform, then load | Put the orders in a warehouse table. Ask it one SQL question |
| 5. Every morning | Run it again the next day. Watch an old number change |
| 6. Why batch, and where it breaks | Measure how late the report is. Then watch the load fail |
| 7. The production version | The same logic as a tested program |

In notebook 1, one call gave you 25 orders. Arden has taken more than 25 orders. This notebook gets all of them.

In [1]:
from lab_helpers import api, bars, set_morning, show_doc

set_morning("2026-06-08")

It is now Monday 2026-06-08 07:00:00 in Arden.


## Step 1: One page is not everything

An API returns rows in **pages**. Ask once, and you get the first page. The answer also tells you how to ask for the next one.

In [2]:
page_1 = api.get("/orders").json()

print("Orders on page 1:", len(page_1["data"]))
print("next_cursor:     ", page_1["next_cursor"])

Orders on page 1: 25
next_cursor:      MjAyNi0wNi0wMlQxNDo0ODowMFp8MTAyNw==


The `next_cursor` is a bookmark. It means "this is where you stopped". Hand it back, and the API gives the page after it.

## Step 2: Follow the bookmark by hand

Do it twice by hand before you write a loop.

In [3]:
page_2 = api.get("/orders", params={"cursor": page_1["next_cursor"]}).json()
page_3 = api.get("/orders", params={"cursor": page_2["next_cursor"]}).json()

for number, page in enumerate([page_1, page_2, page_3], start=1):
    changed = [order["updated_at"] for order in page["data"]]
    print(f"Page {number}: {len(changed)} orders, last changed from {changed[0]} to {changed[-1]}")

Page 1: 25 orders, last changed from 2026-06-01T14:45:06Z to 2026-06-02T14:48:00Z
Page 2: 25 orders, last changed from 2026-06-02T14:49:00Z to 2026-06-03T02:28:19Z
Page 3: 25 orders, last changed from 2026-06-03T02:33:23Z to 2026-06-03T20:57:20Z


Three calls, 75 orders. The API sorts orders by `updated_at`, the time each one last changed. Each page starts where the last one stopped.

How do you know when to stop? The design says:

In [4]:
show_doc("LLD.md", "3. Extract (`extract.py`)")

**From `docs/LLD.md`**

> ## 3. Extract (`extract.py`)
> 
> *(There is a diagram here. Open `docs/LLD.md` to see it.)*
> 
> | Function | Returns |
> |---|---|
> | `read_run_date(client)` | The local date on the API's clock. This is the morning the run is for |
> | `fetch_all_orders(client)` | `Extract(orders, api_calls)`. Raises `RateLimited` on a 429 |
> 
> This is a **full load**: it reads every row on every run.

The last page has no `next_cursor`. That is the stop signal.

## Step 3: The loop

Now the same thing as a loop. Read it line by line. It is the heart of this notebook.

In [5]:
set_morning("2026-06-08")  # start the morning again, so the count of calls is clean

orders = []  # every order we have read so far
cursor = None  # the bookmark; empty before the first call
calls = 0

while True:
    params = {"cursor": cursor} if cursor else None
    body = api.get("/orders", params=params).json()
    calls += 1
    orders += body["data"]
    cursor = body["next_cursor"]
    print(f"Call {calls:>2}: got {len(body['data']):>2} orders. Total so far: {len(orders)}")
    if cursor is None:
        break

print(f"\nDone. {len(orders)} orders in {calls} calls.")

It is now Monday 2026-06-08 07:00:00 in Arden.
Call  1: got 25 orders. Total so far: 25
Call  2: got 25 orders. Total so far: 50
Call  3: got 25 orders. Total so far: 75
Call  4: got 25 orders. Total so far: 100
Call  5: got 25 orders. Total so far: 125
Call  6: got 25 orders. Total so far: 150
Call  7: got 25 orders. Total so far: 175
Call  8: got 25 orders. Total so far: 200
Call  9: got 25 orders. Total so far: 225
Call 10: got  1 orders. Total so far: 226

Done. 226 orders in 10 calls.


Check it. Every order should be there exactly once.

In [6]:
ids = [order["order_id"] for order in orders]

print("Orders read:      ", len(ids))
print("Different orders: ", len(set(ids)))
print("Any order twice?  ", len(ids) != len(set(ids)))

Orders read:       226
Different orders:  226
Any order twice?   False


## Step 4: A new word, full load

You just read **every order Arden has ever taken**. That has a name.

| Word | Meaning | In this notebook |
|---|---|---|
| **Full load** | "A full load reads every row on every run." | 226 orders, in 10 calls |

The definition is from the D8Loop chapter [Data Sources & Ingestion Patterns](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-sources-and-ingestion-patterns).

A full load is the simplest load there is. It needs no memory of the last run. It is also always complete.

### Compare it with its opposite

The opposite is an **incremental load**: read only the rows that are new or changed since the last run.

You will build that in milestone 2. For now, just measure what it would save. Every order has an `updated_at`: the last time it changed. Yesterday's run was at 07:00 on Sunday, which is `2026-06-07T12:00:00Z`. Count the orders that changed after that.

In [7]:
last_run = "2026-06-07T12:00:00Z"
changed = [order for order in orders if order["updated_at"] >= last_run]

bars(
    {
        "Full load reads": len(orders),
        "Changed since yesterday's run": len(changed),
    },
    unit=" orders",
)

Full load reads                ████████████████████████████████████████ 226 orders
Changed since yesterday's run  ██████ 36 orders


Most of what the full load reads is the same as yesterday. Today that costs nothing you can feel. Remember this picture. It comes back at the end of the milestone.

## What you now know

- An API gives data in pages. You follow `next_cursor` until it is empty.
- A full load reads every row on every run.
- On the morning of 8 June, that is 226 orders in 10 calls.
- Only 36 of those orders changed since yesterday's run.

**Say it in an interview:** "My first version was a full load: it read every order on every run by following the API's cursor to the last page. It is simple and always complete, and I measured that most of the rows it read had not changed."

Next: [notebook 3](03_the_answer_by_hand.ipynb) answers the manager's question from these orders, by hand.